# Week 02 · Day 01 — Advanced RAG

**Topics:** CrossEncoder reranking · HyDE · Multi-query retrieval · Contextual compression


In [ ]:
%pip install -q openai chromadb sentence-transformers langchain langchain-openai langchain-community

In [ ]:
import os
from openai import OpenAI
import chromadb

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
chroma = chromadb.Client()

def embed(texts: list[str]) -> list[list[float]]:
    r = client.embeddings.create(input=texts, model="text-embedding-3-small")
    return [item.embedding for item in r.data]

# Build a small knowledge base
docs = [
    "RAG (Retrieval-Augmented Generation) combines information retrieval with language model generation.",
    "Vector databases store high-dimensional embeddings and support approximate nearest neighbor search.",
    "ChromaDB is an open-source embedding database that runs locally or as a server.",
    "FAISS (Facebook AI Similarity Search) is a library for efficient similarity search.",
    "Cosine similarity measures the angle between two vectors, not their magnitude.",
    "BM25 is a ranking function for keyword search used in Elasticsearch and Lucene.",
    "Hybrid search combines dense vector search with sparse keyword search using RRF.",
    "Chunking splits documents into smaller pieces that fit within the LLM context window.",
    "Reranking is a second-pass relevance scoring step that improves retrieval precision.",
    "HyDE generates a hypothetical answer to a query and embeds it for improved retrieval.",
]

col = chroma.get_or_create_collection("adv_rag")
col.add(documents=docs, embeddings=embed(docs), ids=[f"d{i}" for i in range(len(docs))])
print(f"Indexed {col.count()} documents")

## 1 · Baseline Retrieval (Dense-Only)

In [ ]:
def dense_retrieve(query: str, n: int = 5) -> list[str]:
    q_emb = embed([query])[0]
    results = col.query(query_embeddings=[q_emb], n_results=n)
    return results["documents"][0]

query = "how does similarity search work for text?"
baseline = dense_retrieve(query)
print(f"Baseline top-5 for: '{query}'")
for i, doc in enumerate(baseline, 1):
    print(f"  {i}. {doc[:80]}")

## 2 · CrossEncoder Reranking

The bi-encoder (embedding model) retrieves quickly but lacks fine-grained relevance. A CrossEncoder scores (query, doc) pairs jointly — slower but more accurate. Strategy: retrieve top-20, rerank to top-5.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")  # downloads ~22MB

def reranked_retrieve(query: str, retrieve_k: int = 10, final_k: int = 3) -> list[tuple[float, str]]:
    # Step 1: broad retrieval
    candidates = dense_retrieve(query, n=retrieve_k)
    
    # Step 2: score (query, doc) pairs jointly
    pairs = [(query, doc) for doc in candidates]
    scores = reranker.predict(pairs)
    
    # Step 3: return top-k by reranker score
    ranked = sorted(zip(scores, candidates), reverse=True)
    return ranked[:final_k]

print(f"After reranking — top-3 for: '{query}'")
for score, doc in reranked_retrieve(query):
    print(f"  {score:.3f} — {doc[:80]}")

## 3 · HyDE — Hypothetical Document Embeddings

Problem: a short query like "similarity search" doesn't look like the documents in the index. HyDE generates a hypothetical answer (which looks like a document) and embeds that instead.

In [ ]:
def hyde_retrieve(query: str, n: int = 5) -> list[str]:
    # Generate a hypothetical answer
    hypo = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"Write a short technical paragraph that directly answers this question: {query}"
        }],
        temperature=0,
        max_tokens=150,
    ).choices[0].message.content

    # Embed the hypothetical answer (not the query)
    hypo_emb = embed([hypo])[0]
    results = col.query(query_embeddings=[hypo_emb], n_results=n)
    return results["documents"][0], hypo

hyde_results, hypothetical = hyde_retrieve(query)
print(f"Hypothetical doc:\n  {hypothetical[:200]}\n")
print(f"HyDE top-5 for: '{query}'")
for i, doc in enumerate(hyde_results, 1):
    print(f"  {i}. {doc[:80]}")

## 4 · Multi-Query Retrieval

A single query may miss relevant documents phrased differently. Generate multiple paraphrases, retrieve for each, and merge with deduplication.

In [ ]:
def generate_sub_queries(query: str, n: int = 3) -> list[str]:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"Generate {n} different phrasings of this question for a search engine, one per line:\n{query}"
        }],
        temperature=0.7,
        max_tokens=150,
    )
    lines = r.choices[0].message.content.strip().split("\n")
    return [l.lstrip("0123456789.-) ") for l in lines if l.strip()][:n]

def multi_query_retrieve(query: str, n_queries: int = 3, n_results: int = 3) -> list[str]:
    sub_queries = [query] + generate_sub_queries(query, n=n_queries - 1)
    print(f"Sub-queries:")
    for sq in sub_queries:
        print(f"  - {sq}")

    # Retrieve for each sub-query
    seen = set()
    merged = []
    for sq in sub_queries:
        candidates = dense_retrieve(sq, n=n_results)
        for doc in candidates:
            if doc not in seen:
                seen.add(doc)
                merged.append(doc)
    return merged

print(f"Multi-query retrieval for: '{query}'\n")
results = multi_query_retrieve(query)
print(f"\nUnique docs retrieved: {len(results)}")
for i, doc in enumerate(results, 1):
    print(f"  {i}. {doc[:80]}")

## 5 · Contextual Compression

Raw chunks often contain irrelevant sentences. Compression extracts only the relevant portions before passing to the LLM — reduces token cost and improves answer quality.

In [ ]:
def compress_context(query: str, chunks: list[str]) -> list[str]:
    compressed = []
    for chunk in chunks:
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": (
                    f"Extract only the sentences from this text that are directly relevant to the question. "
                    f"If nothing is relevant, respond with 'IRRELEVANT'.\n\n"
                    f"Question: {query}\n\nText: {chunk}"
                )
            }],
            temperature=0,
            max_tokens=100,
        ).choices[0].message.content
        if r != "IRRELEVANT":
            compressed.append(r)
    return compressed

raw_chunks = dense_retrieve(query, n=5)
compressed = compress_context(query, raw_chunks)

raw_tokens = sum(len(c.split()) for c in raw_chunks)
compressed_tokens = sum(len(c.split()) for c in compressed)

print(f"Raw chunks: {len(raw_chunks)} chunks, ~{raw_tokens} words")
print(f"Compressed: {len(compressed)} chunks, ~{compressed_tokens} words")
print(f"Reduction: {(1 - compressed_tokens/raw_tokens)*100:.0f}%")

## Summary

| Technique | When to use | Cost |
|-----------|-------------|------|
| **Reranking** | Always — cheap, consistent gain | +1 model inference |
| **HyDE** | Short queries, sparse documents | +1 LLM call |
| **Multi-query** | Ambiguous or broad queries | +N LLM calls |
| **Compression** | Long chunks, token budget tight | +N LLM calls |

Stack them in order: multi-query → retrieve broad → rerank → compress → generate.

**Next:** [Fine-Tuning](week02-day02-fine-tuning.ipynb)